Tutorial written by Marcus Pearlman on 10/31/2025

# Parallelize
An iterator method is a function that represents one iteration. The following is a function that adds 1 to arg0 if arg1 is `True`.

In [1]:
def addOne(arg0,arg1):
    if arg1:
        return arg0 + 1
    else:
        return arg0 

In [2]:
addOne(2,True)

3

Now we want to run this function multiple times on a list of values. This particular function is something that is easily parallelizable and takes multiple args.

## Traditional Approach
To run this function on multiple values, make a for loop. This is the simplest implementation and is serial.

In [3]:
def loopedAddOne(args0,arg1):
    result = []
    for arg0 in args0:
        result = result + [addOne(arg0,arg1)]
    return result

In [4]:
values = [1,2,3,4]
loopedAddOne(values,True)

[2, 3, 4, 5]

To make this loop run parallely, the list is split between the cores and each core loops through their list. We utilize the multiprocessing package `Pool` class.

In [5]:
from multiprocessing import Pool
import numpy as np

def parallelAddOne(args0,arg1,nc=1):
    steps = args0
    nc = min(nc,len(args0))   # dont use more cores than args0
    if nc>1:
        if type(steps) is range: steps=np.array(steps)     # check if a range() type
        splitArgs0 = np.array_split(args0, nc)             # split list in equal-ish parts
        # for multiple args, we populate a list of tuples for each method call to each core
        variables = [(args0,arg1) for args0 in splitArgs0] 
        pool = Pool(processes=nc)
        result = pool.starmap_async(loopedAddOne,variables)
        result.wait()
        result = result.get()
        pool.close()
        result = np.concatenate(result).tolist()  # make one list from list of lists
    else:
        result = loopedAddOne(args0,arg1)
    return result

This method takes the same args as `addOne` but adds an argument `nc`, which is the number of cores (I don't use `np` for number of processes because `numpy` is `np`).

Let's run this on 4 cores.

In [6]:
parallelAddOne(values,True,nc=4)

[2, 3, 4, 5]

An import aspect of this is that out iterator method takes multiple args. We need to fill in those values for each method call. This is the variables line does.

In [7]:
valuess = np.array_split(values, 4)             # split list in equall-ish parts
# for multiple args, we populate a list of tuples for each method call to each core
variables = [(values,True) for values in valuess] 
print(variables)

[(array([1]), True), (array([2]), True), (array([3]), True), (array([4]), True)]


**_NOTE_** This is important for the parallelize wrapper implementation for an arbitrary number of args

For every iterator method, we would have to create a looped and parallel methods, and have to write this code every time. Now we will create wrappers for this approach so that we don't have to repeat code.

## Wrapped Approach
We first create a looperize method that wraps an arbitrary iterator method with a looped approach.

In [8]:
import inspect
import functools
def looperize(func):
    sig = inspect.signature(func)          # This gets the ordering of the inputs
    @functools.wraps(func)                 # this decorates the function with wrap features?
    def wrapper(*args,**kwargs):
        bound = sig.bind(*args, **kwargs)  # This insures correct ordering
        bound.apply_defaults()             #  This insures correct ordering       
        result = []
        steps = args[0]                    # first argument is the iterable
        for step in args[0]:
            result = result + [func(step,*bound.args[1:],**bound.kwargs)]
        return result
    return wrapper

This method takes a function as an input, and returns a function but now the function is looped. This is because functions are also objects that can be passed around into other functions and wrapped like this. 
The `inspect` modules sig commands binds the args and kwargs to the wrapped function so that `arg` and `kwarg` ordering the input doesn't matter. This makes it more difficult for the user to break your code if they input arguments in the incorrect order.

Let's see if it works.

In [9]:
loopedAddOne = looperize(addOne)
loopedAddOne(values,True)

[2, 3, 4, 5]

It worked! 

**_NOTE_** The first argument must be an `arg`, and not a `kwarg`. `args` are function inputs without the keyword. For example: `addOne(5,True)`, both inputs are `args`. `kwargs` are keyword `args`: inputs with the keyword. For example: `addOne(arg0=5,arg1=True)`, both inputs are `kwargs`. In this looperize wrapper, the first argument is the iterable, and it must be an `arg` (NOT a `kwarg`).

lets try with kwargs

In [10]:
loopedAddOne(arg0=values,arg1=True)

IndexError: tuple index out of range

The error occurs where we try to access args[0], args is empty!!!

Now let's wrap a parallel method out of looped method.

In [11]:
def parallelize_looped_method(func):
    sig = inspect.signature(func)
    @functools.wraps(func)
    def wrapper(*args,**kwargs):
        if 'nc' in kwargs.keys():
            nc = kwargs.pop('nc')
        else:
            nc=1
        
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()
        
        # first argument is the iterable
        steps = bound.args[0]
        nc = min(nc,len(steps))   # dont use more cores than steps
        if nc>1:
            if type(steps) is range: steps=np.array(steps)
            stepss = np.array_split(steps, nc)
            variables = [(steps,)+bound.args[1:]+tuple(bound.kwargs.values()) for steps in stepss]
            pool = Pool(processes=nc)
            result = pool.starmap_async(func,variables)
            result.wait()
            result = result.get()
            pool.close()
            
            result = np.concatenate(result).tolist()  # make one list from list of lists
        else:
            result = func(steps,*args[1:],**kwargs)
        
        return result
    return wrapper

We will first run the code with nc=1

In [12]:
parallelAddOne = parallelize_looped_method(loopedAddOne)
parallelAddOne(values,True,nc=1)

[2, 3, 4, 5]

This works, now lets try with nc=4

In [13]:
parallelAddOne(values,True,nc=4)

PicklingError: Can't pickle <function addOne at 0x7310f87e6e80>: it's not the same object as __main__.addOne

What Happened!!! We cant pickle!? 

So what's happening here is that multiprocessing `Pool` uses another package `pickle` to serialize the data to pass to each core. The short answer is that `pickle` can't serialize complicated wrapped methods. So what do we do? We use another package, `dill`, a more robust pickler. Well, actually, we use a fork of the multiprocessing package that uses `dill`: `multiprocess`.

In [14]:
from multiprocess import Pool
parallelAddOne(values,True,nc=4)

[2, 3, 4, 5]

Now it works! Phew, that was an easy fix. and apparently we don't need to redefine `parallelAddOne` after importing the new `Pool` class, the previously defined function will use the newly imported `Pool` class...

Finally, we can create a wrapper that double wraps an iterator method into a parallel method.

In [15]:
def parallelize_iterator_method(func):
    func = looperize(func)
    return parallelize_looped_method(func)

This utilizes both of our wrappers to make a parallel implementation of any iterator method

In [16]:
parallelAddOne = parallelize_iterator_method(addOne)
parallelAddOne(values,True,nc=4)

[2, 3, 4, 5]

Viola!

## Dmanage Implementation

To do this in Dmanage, we utilize the `methods.wrapper` package. Typically, I define the iterator methods as a private method, using the preceding underscore '_'. Even though we don't need to redefine a new function and just call `addOne = wrapper.parallelize_iterator_method(_addOne)`, I like to jazz up my method with timing printouts.

In [17]:

from dmanage import parallel
from dmanage.utils.objinfo import is_iterable
import time

def _addOne(arg0,arg1):
    if arg1:
        return arg0 + 1
    else:
        return arg0 

def addOne(arg0,arg1,nc=1):
    addOne = wrapper.parallelize_iterator_method(_addOne)
    startTime = time.time()
    if arg1:
        if not is_iterable(arg0): arg0 = [arg0]   # determine if it is an iterable and make it one
        nc = min(nc,len(arg0))
        print('Adding one to values using %i cores...'%(nc), end=' ')
        result = addOne(arg0,arg1,nc=nc)
        executionTime = (time.time()-startTime)
        print(' Done in %0.2f seconds'%(executionTime))
        return result
    else:
        return arg0

now when we call our addOne method, it prints out some diagnostics

In [18]:
addOne(values,arg1=True,nc=4)

Adding one to values using 4 cores...  Done in 0.02 seconds


[np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

**__NOTE__** The parallel method takes longer in this case! This is because the problem size (only 4 elements) is too small to outweigh the overhead cost to set up the processing pool. With bigger problem sizes, the parallel method will be faster.

The parallelize wrappers also attempts to coerce the `args0` input into the proper data type. It can take a scaler value, and any iterator type such as `list`, `tuples`, `range`, `np.array`. The `isIterable()` method determines if an object is an iterable and not a `str`, which is technically an iterable.

In [19]:
addOne(1,arg1=True,nc=4)

Adding one to values using 1 cores...  Done in 0.00 seconds


[2]

In [20]:
addOne((1,2,3,4),arg1=True,nc=4)

Adding one to values using 4 cores...  Done in 0.02 seconds


[np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

In [21]:
addOne(np.array([1,2,3,4]),arg1=True,nc=4)

Adding one to values using 4 cores...  Done in 0.03 seconds


array([2, 3, 4, 5])